# Debug: #470 — rows shift down under multi-row headers

**Symptom:** with a multi-row column header, the body rows under the 'controllability' columns get shifted down ~2 rows. The lattice grid is detected correctly (per the issue's screenshots) — it's the text→cell *assignment* that's off.

**Hypothesis:** `get_table_index` assigns a textline to the row whose midpoint contains the textline's y-midpoint. When a merged multi-row header pushes that midpoint into an adjacent grid cell, body rows shift. The fix is likely to change the heuristic from 'midpoint-in-row-range' to 'bbox-top closest to row-top'.

**This notebook is exploratory** — there's no proposed patch yet. The goal is to (a) reproduce, (b) confirm the grid is right but assignment is wrong, (c) prototype the row-assignment change.

Issue: [#470](https://github.com/camelot-dev/camelot/issues/470) · **Setup:** `pip install -e .[plot,base]`.

In [ ]:
import urllib.request, os
os.makedirs('/tmp/camelot-470', exist_ok=True)
dst = '/tmp/camelot-470/118.pdf'
if not os.path.exists(dst):
    try:
        urllib.request.urlretrieve('https://poetaster.de/misc/118.pdf', dst)
        print('downloaded', dst)
    except Exception as e:
        print('download failed:', e, '\n manually fetch https://poetaster.de/misc/118.pdf ->', dst)
else:
    print('have', dst)

## Step 1 — reproduce + confirm the grid is correct

Extract with lattice, then plot the grid to confirm cell boundaries are detected right (the issue says they are). If the grid is right but the dataframe is shifted, the bug is purely in text assignment.

In [ ]:
import camelot
tables = camelot.read_pdf('/tmp/camelot-470/118.pdf', flavor='lattice', line_scale=40)
print('tables:', len(tables))
for i, t in enumerate(tables):
    print(f'table {i}: shape={t.shape}')
tables[0].df if tables else None

In [ ]:
# Plot the detected grid — confirm cell boundaries match the visual table.
import camelot
camelot.plot(tables[0], kind='grid').show()  # needs matplotlib

## Step 2 — try the existing escape hatches

Before touching the algorithm, check whether `flavor='hybrid'` or `copy_text=['v']` already produces the right rows. If either does, the issue can be answered with a docs note instead of a code change.

In [ ]:
for variant, kw in [
    ('hybrid', dict(flavor='hybrid')),
    ('lattice+copy_text v', dict(flavor='lattice', copy_text=['v'])),
    ('lattice+line_scale 40', dict(flavor='lattice', line_scale=40)),
]:
    try:
        ts = camelot.read_pdf('/tmp/camelot-470/118.pdf', **kw)
        print(f'--- {variant}: {len(ts)} table(s) ---')
        if ts:
            print(ts[0].df.head(8).to_string())
    except Exception as e:
        print(f'--- {variant}: ERROR {e} ---')
    print()

## Step 3 — prototype the row-assignment change

If the escape hatches don't fix it, the change is in `camelot/utils.py` `get_table_index` / `_process_horizontal_cut` — the row-index computation:

```python
r_idx = [j for j, r in enumerate(table.rows)
         if r[1] <= (bbox[1] + bbox[3]) / 2 <= r[0]]   # midpoint-in-range
```

Prototype: assign to the row whose TOP edge (`r[0]`) is closest to the textline's top (`bbox[3]`), which is more robust when a textline sits inside a tall merged header cell. Monkey-patch and re-extract to compare.

In [ ]:
import camelot.utils as U, inspect
print(inspect.getsource(U._process_horizontal_cut))
# Use this as the base for the monkey-patched row-assignment prototype.